# Create Boxplots Showing the Distribution of Generation by Month


In [9]:
# Start by importing the packages we need:
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Suppress Future Warnings


In [10]:
# Suppress future warnings:
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)


## Set the Directory Structure

In [11]:
# Set the data input and output directories:
hw_cs_data_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/thermal_events_data/'
integrated_time_series_data_input_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/integrated_time_series/'
image_output_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/figures/regional_results/'


## Plot the Generation Mix by Month


In [65]:
def plot_generation_mix(region: str, hw_cs_data_dir: str, integrated_time_series_data_input_dir: str, image_output_dir: str, image_resolution: int, save_images=False):

    # Subset to just the data for region you want to use:
    if region == 'CA':
       region_name = 'California'
    if region == 'GB':
       region_name = 'Great Basin'
    
    # Read in the heat wave event library and subset to just the region being plotted:
    hw_df = pd.read_csv((hw_cs_data_dir + 'hw_library_expanded.csv'))
    cs_df = pd.read_csv((hw_cs_data_dir + 'cs_library_expanded.csv'))
    hw_df = hw_df[(hw_df['Region'] == region)].copy()
    cs_df = cs_df[(cs_df['Region'] == region)].copy()

    # Read in the integrated time series:
    gen_df = pd.read_csv((integrated_time_series_data_input_dir + region + '_Integrated_Time_Series_1980_to_2024.csv'))

    # Extract the month and date for grouping:
    gen_df['Time_UTC'] = pd.to_datetime(gen_df['Time_UTC'])
    gen_df['Month'] = gen_df['Time_UTC'].dt.month
    gen_df['Date'] = gen_df['Time_UTC'].dt.date
    gen_df['Date'] = pd.to_datetime(gen_df['Date'])

    # Calculate the total generation across all types and for renewable vs non-renewable:
    gen_df['Total'] = gen_df['Coal'] + gen_df['Gas'] + gen_df['Hydro'] + gen_df['Other'] + gen_df['Solar'] + gen_df['Wind']
    gen_df['Fossil'] = gen_df['Coal'] + gen_df['Gas']
    gen_df['Renewable'] = gen_df['Hydro'] + gen_df['Solar'] + gen_df['Wind'] + gen_df['Other']

    # Subset to just the days with valid generation data:
    gen_df = gen_df.loc[(gen_df['Total'] >= 0) & (gen_df['Region_Load'] >= 0)].copy()
    
    # Loop over rows in the heat wave event library and extract the time series for just the events:
    for row in range(len(hw_df)):
        # Extract the start and end dates for the event in that row:
        start_date = pd.to_datetime(hw_df['Start'].iloc[[row]].item())
        end_date = pd.to_datetime(hw_df['End'].iloc[[row]].item())

        # Subset to just the hours during that event:
        temp_df = gen_df[(gen_df['Date'] >= start_date) & (gen_df['Date'] <= end_date)]

        # Aggregate the output into a new dataframe:
        if row == 0:
           hw_gen_df = temp_df
        else:
           hw_gen_df = pd.concat([hw_gen_df, temp_df])

        # Clean up and move to the next row:
        del start_date, end_date, temp_df

    # Loop over rows in the cold snap event library and extract the time series for just the events:
    for row in range(len(cs_df)):
        # Extract the start and end dates for the event in that row:
        start_date = pd.to_datetime(cs_df['Start'].iloc[[row]].item())
        end_date = pd.to_datetime(cs_df['End'].iloc[[row]].item())

        # Subset to just the hours during that event:
        temp_df = gen_df[(gen_df['Date'] >= start_date) & (gen_df['Date'] <= end_date)]

        # Aggregate the output into a new dataframe:
        if row == 0:
           cs_gen_df = temp_df
        else:
           cs_gen_df = pd.concat([cs_gen_df, temp_df])

        # Clean up and move to the next row:
        del start_date, end_date, temp_df
        
    # Subset to just the daytime values for solar:
    day_gen_df = gen_df[gen_df['Solar'] != 0]
    day_hw_gen_df = hw_gen_df[hw_gen_df['Solar'] != 0]
    day_cs_gen_df = cs_gen_df[cs_gen_df['Solar'] != 0]
    
    # Make the plot:
    plt.figure(figsize=(30,35))
    plt.rcParams['font.size'] = 18
    plt.rcParams['axes.axisbelow'] = True
    
    ax1 = plt.subplot(611)
    plt.plot([np.nan,np.nan], color='b', linestyle='-', label='Cold Snap Days', linewidth=2)
    plt.plot([np.nan,np.nan], color='k', linestyle='-', label='All Days', linewidth=2)
    plt.plot([np.nan,np.nan], color='r', linestyle='-', label='Heat Wave Days', linewidth=2)
    for month in range(1,13):
        box1a = plt.boxplot(gen_df['Region_Load'].loc[(gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box1a[item], color='k', markeredgecolor='k', linewidth=2)
        box1b = plt.boxplot(hw_gen_df['Region_Load'].loc[(hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box1b[item], color='r', markeredgecolor='r', linewidth=2)   
        box1c = plt.boxplot(cs_gen_df['Region_Load'].loc[(cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box1c[item], color='b', markeredgecolor='b', linewidth=2)    
    plt.legend(loc='upper left', prop={'size': 12})
    plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.xlim([0.5, 12.5])
    plt.ylabel('Hourly Demand [MWh]', fontsize=18)
    plt.title(('Total Demand in ' + region_name + ' Region'))
    plt.title('a)', loc='left', fontsize=16)

    #ax1 = plt.subplot(611)
    #plt.plot([np.nan,np.nan], color='b', linestyle='-', label='Cold Snap Days', linewidth=2)
    #plt.plot([np.nan,np.nan], color='k', linestyle='-', label='All Days', linewidth=2)
    #plt.plot([np.nan,np.nan], color='r', linestyle='-', label='Heat Wave Days', linewidth=2)
    #for month in range(1,13):
    #    box1a = plt.boxplot(gen_df['Total'].loc[(gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
    #    for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
    #        plt.setp(box1a[item], color='k', markeredgecolor='k', linewidth=2)
    #    box1b = plt.boxplot(hw_gen_df['Total'].loc[(hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
    #    for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
    #        plt.setp(box1b[item], color='r', markeredgecolor='r', linewidth=2)   
    #    box1c = plt.boxplot(cs_gen_df['Total'].loc[(cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
    #    for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
    #        plt.setp(box1c[item], color='b', markeredgecolor='b', linewidth=2)    
    #plt.legend(loc='upper left', prop={'size': 12})
    #plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    #plt.xlim([0.5, 12.5])
    #plt.ylabel('Hourly Generation [MWh]', fontsize=18)
    #plt.title(('Total Generation in ' + region_name + ' Region'))
    #plt.title('a)', loc='left', fontsize=16)

    ax2 = plt.subplot(612)
    for month in range(1,13):
        box2a = plt.boxplot(gen_df['Fossil'].loc[(gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box2a[item], color='k', markeredgecolor='k', linewidth=2)
        box2b = plt.boxplot(hw_gen_df['Fossil'].loc[(hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box2b[item], color='r', markeredgecolor='r', linewidth=2)
        box2c = plt.boxplot(cs_gen_df['Fossil'].loc[(cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box2c[item], color='b', markeredgecolor='b', linewidth=2)    
    plt.ylim(bottom=0)
    plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.xlim([0.5, 12.5])
    plt.ylabel('Hourly Generation [MWh]', fontsize=18)
    plt.title(('Coal + Gas Generation'))
    plt.title('b)', loc='left', fontsize=16)

    ax3 = plt.subplot(613)
    for month in range(1,13):
        box3a = plt.boxplot(gen_df['Hydro'].loc[(gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box3a[item], color='k', markeredgecolor='k', linewidth=2)
        box3b = plt.boxplot(hw_gen_df['Hydro'].loc[(hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box3b[item], color='r', markeredgecolor='r', linewidth=2)
        box3c = plt.boxplot(cs_gen_df['Hydro'].loc[(cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box3c[item], color='b', markeredgecolor='b', linewidth=2)
    plt.ylim(bottom=0)
    plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.xlim([0.5, 12.5])
    plt.ylabel('Hourly Generation [MWh]', fontsize=18)
    plt.title(('Hydropower Generation'))
    plt.title('c)', loc='left', fontsize=16)

    ax4 = plt.subplot(614)
    for month in range(1,13):
        box4a = plt.boxplot(gen_df['Wind'].loc[(gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box4a[item], color='k', markeredgecolor='k', linewidth=2)
        box4b = plt.boxplot(hw_gen_df['Wind'].loc[(hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box4b[item], color='r', markeredgecolor='r', linewidth=2)
        box4c = plt.boxplot(cs_gen_df['Wind'].loc[(cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box4c[item], color='b', markeredgecolor='b', linewidth=2)
    plt.ylim(bottom=0)
    plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.xlim([0.5, 12.5])    
    plt.ylabel('Hourly Generation [MWh]', fontsize=18)
    plt.title(('Wind Generation'))
    plt.title('d)', loc='left', fontsize=16)
    
    ax5 = plt.subplot(615)
    for month in range(1,13):
        box5a = plt.boxplot(day_gen_df['Solar'].loc[(day_gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box5a[item], color='k', markeredgecolor='k', linewidth=2)
        box5b = plt.boxplot(day_hw_gen_df['Solar'].loc[(day_hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box5b[item], color='r', markeredgecolor='r', linewidth=2)
        box5c = plt.boxplot(day_cs_gen_df['Solar'].loc[(day_cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box5c[item], color='b', markeredgecolor='r', linewidth=2)    
    plt.ylim(bottom=0)
    plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.xlim([0.5, 12.5])    
    plt.ylabel('Hourly Generation [MWh]', fontsize=18)
    plt.title(('Solar Generation (Daytime Values Only)'))
    plt.title('e)', loc='left', fontsize=16)

    ax6 = plt.subplot(616)
    for month in range(1,13):
        box6a = plt.boxplot(gen_df['Imports'].loc[(gen_df['Month']==month)], vert=True, positions=[month], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box6a[item], color='k', markeredgecolor='k', linewidth=2)
        box6b = plt.boxplot(hw_gen_df['Imports'].loc[(hw_gen_df['Month']==month)], vert=True, positions=[month+0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box6b[item], color='r', markeredgecolor='r', linewidth=2)
        box6c = plt.boxplot(cs_gen_df['Imports'].loc[(cs_gen_df['Month']==month)], vert=True, positions=[month-0.2], widths=0.2)
        for item in ['boxes', 'whiskers', 'fliers', 'medians', 'caps']:
            plt.setp(box6c[item], color='b', markeredgecolor='b', linewidth=2)
    #plt.ylim(bottom=0)
    plt.plot([0,13], [0,0],'gray', linewidth=2)    
    plt.xticks([1,2,3,4,5,6,7,8,9,10,11,12],['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
    plt.xlim([0.5, 12.5])    
    plt.ylabel('Hourly Imports [MWh]', fontsize=18)
    plt.title(('Imports From Other Regions'))
    plt.title('f)', loc='left', fontsize=16)

    # If the "save_images" flag is set to true then save the plot to a .png file:
    if save_images == True:
       plt.savefig(os.path.join(image_output_dir + region + '_Annual_Generation.png'), dpi=image_resolution, bbox_inches='tight')
       plt.close()

    return gen_df


In [68]:
# Test the function:
output_df = plot_generation_mix(region = 'CA',
                                hw_cs_data_dir = hw_cs_data_dir,
                                integrated_time_series_data_input_dir = integrated_time_series_data_input_dir,
                                image_output_dir = image_output_dir, 
                                image_resolution = 150,
                                save_images = True)

# output_df
